In [1]:
import pyspark

In [2]:
print(pyspark.__version__)

3.5.1


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BigDataPractise")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

In [4]:
spark

In [5]:
# Load the CSV data
data = spark.read.csv("sample_data.csv", header = True, inferSchema = True)

In [6]:
data.show()

+--------+--------------------+----+-------+
|    Name|           Job_title| Age| Salary|
+--------+--------------------+----+-------+
|  Deepak|      Senior_Analyst|  32| 800000|
|    Arun|         AI_Engineer|  30|1000000|
|Priyanka|          ML_Trainer|  36| 650000|
|     Ram|         IT_Engineer|  28|   NULL|
|  Nikita|         IT_Engineer|  27| 750000|
|Santhosh|     UI/UX_developer|  30| 850000|
|    Neha|      Data_Scientist|  30|1200000|
|    Riya|                NULL|  26| 600000|
|   Madhu|Full_Stack_Developer|NULL| 950000|
+--------+--------------------+----+-------+



In [7]:
# Datatypes
data.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Job_title: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [8]:
data.dtypes

[('Name', 'string'),
 ('Job_title', 'string'),
 ('Age', 'int'),
 ('Salary', 'int')]

In [9]:
#Shape of the data
print("Rows:", data.count())
print("Columns:", len(data.columns))

Rows: 9
Columns: 4


In [10]:
# Describe
data.describe().show()

+-------+--------+---------------+-----------------+------------------+
|summary|    Name|      Job_title|              Age|            Salary|
+-------+--------+---------------+-----------------+------------------+
|  count|       9|              8|                8|                 8|
|   mean|    NULL|           NULL|           29.875|          850000.0|
| stddev|    NULL|           NULL|3.136763573220926|196396.10121239314|
|    min|    Arun|    AI_Engineer|               26|            600000|
|    max|Santhosh|UI/UX_developer|               36|           1200000|
+-------+--------+---------------+-----------------+------------------+



In [11]:
data.show()

+--------+--------------------+----+-------+
|    Name|           Job_title| Age| Salary|
+--------+--------------------+----+-------+
|  Deepak|      Senior_Analyst|  32| 800000|
|    Arun|         AI_Engineer|  30|1000000|
|Priyanka|          ML_Trainer|  36| 650000|
|     Ram|         IT_Engineer|  28|   NULL|
|  Nikita|         IT_Engineer|  27| 750000|
|Santhosh|     UI/UX_developer|  30| 850000|
|    Neha|      Data_Scientist|  30|1200000|
|    Riya|                NULL|  26| 600000|
|   Madhu|Full_Stack_Developer|NULL| 950000|
+--------+--------------------+----+-------+



In [27]:
#Fill the null values with Mean, Median and Mode
#Fill Salary with mean
from pyspark.ml.feature import Imputer

mean_imputer = Imputer(
    inputCol = 'Salary',
    outputCol = 'Salary_imputed',
    strategy = 'mean'
)

In [28]:
#Fill Age with median
median_imputer = Imputer(
    inputCol = 'Age',
    outputCol = 'Age_imputed',
    strategy = 'median'
)

In [24]:
# Fill Job_Title with Mode
from pyspark.sql.functions import col
mode_Job = (
    data.groupBy("Job_title")
    .count()
    .orderBy(col("count").desc())
    .first()[0]
)           

In [25]:
data = data.fillna({"Job_Title": mode_Job})

In [29]:
data = mean_imputer.fit(data).transform(data)
data = median_imputer.fit(data).transform(data)

In [30]:
data.show()

+--------+--------------------+----+-------+--------------+-----------+
|    Name|           Job_title| Age| Salary|Salary_imputed|Age_imputed|
+--------+--------------------+----+-------+--------------+-----------+
|  Deepak|      Senior_Analyst|  32| 800000|        800000|         32|
|    Arun|         AI_Engineer|  30|1000000|       1000000|         30|
|Priyanka|          ML_Trainer|  36| 650000|        650000|         36|
|     Ram|         IT_Engineer|  28|   NULL|        850000|         28|
|  Nikita|         IT_Engineer|  27| 750000|        750000|         27|
|Santhosh|     UI/UX_developer|  30| 850000|        850000|         30|
|    Neha|      Data_Scientist|  30|1200000|       1200000|         30|
|    Riya|         IT_Engineer|  26| 600000|        600000|         26|
|   Madhu|Full_Stack_Developer|NULL| 950000|        950000|         30|
+--------+--------------------+----+-------+--------------+-----------+

